# VoiceGuard — Fusion Layer Training & Calibration (Google Colab)

This notebook trains and calibrates the multi-signal Bayesian fusion layer combining acoustic detection, linguistic scam classification, and interactive challenge verification.

### Rationale:
- Fits L2-regularised Logistic Regression with Isotonic probability calibration.
- Uses exact 14-dimensional feature vector.
- Compares with gradient boosted trees (`HistGradientBoostingClassifier`).
- Evaluates Brier score, Reliability diagrams, Expected Calibration Error (ECE), and feature contributions.

In [ ]:
# 1. Environment & Setup
!pip install -q scikit-learn>=1.4.0 numpy matplotlib
import sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    print("Cloning VoiceGuard repository from GitHub...")
    !git clone https://github.com/AS24xADITYA/VoiceGuard.git /content/VoiceGuard
    repo_root = Path("/content/VoiceGuard").resolve()

backend_dir = repo_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))
print(f"Loaded VoiceGuard backend from: {backend_dir}")


## 2. Multi-Signal Dataset Matrix Construction

The calibrated fusion layer takes exactly 14 features matching `ai.fusion.features.FEATURE_NAMES`:
1. `acoustic_available` {0, 1}
2. `acoustic_spoof_prob` [0, 1]
3. `acoustic_uncertainty` [0, 1]
4. `acoustic_window_std` [0, 1]
5. `linguistic_available` {0, 1}
6. `scam_prob` [0, 1]
7. `scam_max_category` [0, 1]
8. `scam_n_categories` [0, 1]
9. `transcript_reliable` {0, 1}
10. `language_supported` {0, 1}
11. `transcript_length_norm` [0, 1]
12. `challenge_available` {0, 1}
13. `challenge_consistency` [0, 1]
14. `audio_quality_score` [0, 1]

In [ ]:
# ── 2. Construct 14-Feature Matrix ─────────────────────────────────────
import numpy as np
from ai.fusion.features import FEATURE_NAMES

np.random.seed(42)
n_train = 4000
n_val = 1000

def generate_realistic_features(n_samples):
    # 4 distinct fraud regimes:
    # 0: clean bona fide human
    # 1: acoustic clone (high spoof prob, low scam intent)
    # 2: human extortion scam (low spoof prob, high scam intent)
    # 3: multi-modal attack (high spoof prob, high scam intent)
    X = np.zeros((n_samples, 14))
    y = np.zeros(n_samples, dtype=int)
    
    for i in range(n_samples):
        regime = np.random.choice([0, 1, 2, 3], p=[0.40, 0.25, 0.25, 0.10])
        if regime == 0:  # Bona fide benign
            acoustic_prob = np.random.beta(1.5, 15.0)
            scam_prob = np.random.beta(1.0, 12.0)
            y[i] = 0
        elif regime == 1: # Acoustic deepfake without extortion
            acoustic_prob = np.random.beta(12.0, 2.0)
            scam_prob = np.random.beta(1.0, 10.0)
            y[i] = 1
        elif regime == 2: # Human extortion scam call
            acoustic_prob = np.random.beta(1.5, 12.0)
            scam_prob = np.random.beta(14.0, 2.0)
            y[i] = 1
        else: # Multi-modal clone extortion
            acoustic_prob = np.random.beta(15.0, 1.5)
            scam_prob = np.random.beta(15.0, 1.5)
            y[i] = 1
            
        # 0: acoustic_available
        X[i, 0] = 1.0 if (regime != 2 or np.random.rand() > 0.15) else 0.0
        # 1: acoustic_spoof_prob
        X[i, 1] = acoustic_prob if X[i, 0] == 1.0 else 0.50
        # 2: acoustic_uncertainty
        X[i, 2] = float(np.clip(-(X[i, 1] * np.log2(X[i, 1] + 1e-6) + (1 - X[i, 1]) * np.log2(1 - X[i, 1] + 1e-6)), 0.0, 1.0)) if X[i, 0] == 1.0 else 1.00
        # 3: acoustic_window_std
        X[i, 3] = np.random.uniform(0.01, 0.15) if X[i, 0] == 1.0 else 0.0
        # 4: linguistic_available
        X[i, 4] = 1.0 if (regime != 1 or np.random.rand() > 0.15) else 0.0
        # 5: scam_prob
        X[i, 5] = scam_prob if X[i, 4] == 1.0 else 0.50
        # 6: scam_max_category
        X[i, 6] = scam_prob * np.random.uniform(0.7, 1.0) if X[i, 4] == 1.0 else 0.0
        # 7: scam_n_categories
        X[i, 7] = (np.random.randint(1, 5) / 8.0) if (X[i, 4] == 1.0 and scam_prob > 0.5) else 0.0
        # 8: transcript_reliable
        X[i, 8] = 1.0 if X[i, 4] == 1.0 else 0.0
        # 9: language_supported
        X[i, 9] = 1.0 if X[i, 4] == 1.0 else 0.0
        # 10: transcript_length_norm
        X[i, 10] = np.random.uniform(0.2, 0.8) if X[i, 4] == 1.0 else 0.0
        # 11: challenge_available
        has_challenge = np.random.rand() < 0.35
        X[i, 11] = 1.0 if has_challenge else 0.0
        # 12: challenge_consistency
        if has_challenge:
            X[i, 12] = np.random.uniform(0.75, 0.98) if (regime == 0 or regime == 2) else np.random.uniform(0.1, 0.45)
        else:
            X[i, 12] = 0.50
        # 13: audio_quality_score
        X[i, 13] = np.random.uniform(0.70, 0.98)
        
    return X, y

X_train, y_train = generate_realistic_features(n_train)
X_val, y_val = generate_realistic_features(n_val)
print(f"Generated training matrix matching FEATURE_NAMES: X={X_train.shape}, positive_labels={y_train.sum()}/{len(y_train)}")


In [ ]:
# ── 3. Train and Calibrate Fusion Model ────────────────────────────────
from ai.fusion.train_fusion import train_fusion_model

summary = train_fusion_model(
    x_train=X_train,
    y_train=y_train,
    x_val=X_val,
    y_val=y_val,
    output_dir="/content/fusion_model_output",
)

print("\n--- Calibrated Logistic Regression Coefficients ---")
for feat, coef in summary["logistic_regression"]["coefficients"].items():
    print(f"  {feat:32s}: {coef:+.4f}")
print(f"  {'intercept':32s}: {summary['logistic_regression']['intercept']:+.4f}")


In [ ]:
# ── 4. Export Artifact & 1-Click Browser Download ──────────────────────
import shutil
from pathlib import Path

src_artifact = Path("/content/fusion_model_output/fusion_model.pkl")
dst_artifact = Path("/content/fusion.pkl")

if src_artifact.exists():
    shutil.copy(src_artifact, dst_artifact)
    print(f"✓ Fusion model artifact ready: {dst_artifact}")
    print("\n--- Initiating Browser Download ---")
    try:
        from google.colab import files
        files.download(str(dst_artifact))
        print("✓ Download prompt opened! Place this file in: VoiceGuard/backend/models/fusion.pkl")
    except Exception as e:
        print(f"Download manually from Colab file browser: {dst_artifact}")
else:
    print("Fusion artifact not found. Ensure training step completed.")
